# Experiment 1 analysis

Reads the consolidated CSVs under `data/outputs/Experiment 1/_analysis/` and renders, in order:

1. Runs per category ? model (count of model evaluations).
2. Per-model averaged metrics (AUROC, AUPR, precision, recall, latency ms, F1).
3. Per-category ? model mini-matrix (AUROC | F1 / AUPR | latency).
4. One detailed table per model.
5. Heatmap AUROC (category ? model).
6. Heatmap F1 (category ? model).
7. Offline test-set score scatter from `test_scores.csv`.

The notebook does not read raw experiment output folders.


In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks and aux scripts"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from hardware_equivalence import normalize_latency

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
plt.rcParams["figure.dpi"] = 110

ALL_MODELS = [
    "patchcore", "padim", "subspacead", "stfpm",
    "csflow", "draem", "rd4ad",
]

REAL_IAD_CATEGORIES = [
    "audiojack", "bottle_cap", "button_battery", "end_cap", "eraser",
    "fire_hood", "mint", "mounts", "pcb", "phone_battery",
    "plastic_nut", "plastic_plug", "porcelain_doll", "regulator",
    "rolled_strip_base", "sim_card_set", "switch", "tape",
    "terminalblock", "toothbrush", "toy", "toy_brick", "transistor1",
    "u_block", "usb", "usb_adaptor", "vcpill", "wooden_beads",
    "woodstick", "zipper",
]

EXP_ROOT = PROJECT_ROOT / "data" / "outputs" / "Experiment 1"
ANALYSIS_DIR = EXP_ROOT / "_analysis"
RUNS_CSV = ANALYSIS_DIR / "runs_cache.csv"
SCORES_CSV = ANALYSIS_DIR / "test_scores.csv"

print(f"Runs CSV: {RUNS_CSV}")
print(f"Scores CSV: {SCORES_CSV}")


In [ ]:
df = pd.read_csv(RUNS_CSV)
score_df = pd.read_csv(SCORES_CSV, low_memory=False)
normalize_latency(df)
MODELS_PRESENT = sorted(df["model"].dropna().unique()) if not df.empty else []
print(f"Loaded {len(df)} consolidated run rows across models: {MODELS_PRESENT}")
print(f"Loaded {len(score_df)} consolidated score rows.")


## 1. Runs per category × model

Count of model evaluations found in `Experiment 1/` for each (category, model) pair. The `jobA_val_defect_*` folders contribute three entries each (PatchCore, PaDiM, SubspaceAD) since they share the same training split.

In [ ]:
runs_per_cat = (
    df.groupby(["category", "model"]).size().unstack(fill_value=0)
    .reindex(index=REAL_IAD_CATEGORIES, columns=ALL_MODELS, fill_value=0)
)
runs_per_cat.index.name = "category"
runs_per_cat.loc["TOTAL"] = runs_per_cat.sum(axis=0)
runs_per_cat

## 2. Per-model averaged metrics

For each model, mean across the categories actually run. Columns include every model supported by the pipeline; models with no Experiment 1 outputs show NaN.

In [ ]:
METRICS_AVG = ["auroc", "aupr", "precision", "recall", "mean_latency_ms", "f1"]

model_avg = df.groupby("model")[METRICS_AVG].mean(numeric_only=True).T
model_avg = model_avg.reindex(columns=ALL_MODELS)
model_avg.index.name = "metric"
model_avg

## 3. Category × model — mini-matrix per cell

Each cell is a 2×2 block:

| AUROC | F1 |
| --- | --- |
| **AUPR** | **latency (ms)** |

In [ ]:
def _fmt(v: float, is_latency: bool = False) -> str:
    if pd.isna(v):
        return "\u2014"
    return f"{v:.1f}" if is_latency else f"{v:.3f}"


def render_minimatrix(df: pd.DataFrame, models: list[str], categories: list[str]) -> str:
    aur = df.pivot_table(index="category", columns="model", values="auroc")
    f1m = df.pivot_table(index="category", columns="model", values="f1")
    aup = df.pivot_table(index="category", columns="model", values="aupr")
    lat = df.pivot_table(index="category", columns="model", values="mean_latency_ms")

    def get(pivot, c, m):
        if c in pivot.index and m in pivot.columns:
            return pivot.loc[c, m]
        return float("nan")

    def cell_html(c: str, m: str) -> str:
        cell = (
            "<table style='border-collapse:collapse;font-size:10px;width:100%'>"
            "<tr>"
            f"<td style='padding:1px 4px;border-right:1px solid #ddd;border-bottom:1px solid #ddd'>"
            f"<span style='color:#888'>AUROC</span><br><b>{_fmt(get(aur, c, m))}</b></td>"
            f"<td style='padding:1px 4px;border-bottom:1px solid #ddd'>"
            f"<span style='color:#888'>F1</span><br><b>{_fmt(get(f1m, c, m))}</b></td>"
            "</tr><tr>"
            f"<td style='padding:1px 4px;border-right:1px solid #ddd'>"
            f"<span style='color:#888'>AUPR</span><br><b>{_fmt(get(aup, c, m))}</b></td>"
            f"<td style='padding:1px 4px'>"
            f"<span style='color:#888'>LAT ms</span><br><b>{_fmt(get(lat, c, m), is_latency=True)}</b></td>"
            "</tr></table>"
        )
        return cell

    header = "".join(
        f"<th style='border:1px solid #999;padding:4px;background:#f3f3f3'>{m}</th>"
        for m in models
    )
    body = []
    for c in categories:
        cells_html = "".join(
            f"<td style='border:1px solid #999;padding:2px;vertical-align:top'>{cell_html(c, m)}</td>"
            for m in models
        )
        body.append(
            "<tr>"
            f"<th style='border:1px solid #999;padding:4px;text-align:left;background:#f3f3f3'>{c}</th>"
            f"{cells_html}</tr>"
        )
    return (
        "<table style='border-collapse:collapse'>"
        f"<tr><th style='border:1px solid #999;padding:4px;background:#f3f3f3'>category</th>{header}</tr>"
        + "".join(body)
        + "</table>"
    )


display(HTML(render_minimatrix(df, ALL_MODELS, REAL_IAD_CATEGORIES)))

## 4. Per-model detailed table

One table per model. Rows are categories that were run for that model; columns are the headline run-level metrics. The `train_samples` / `val_samples` / `test_samples` triple replaces Experiment 3's warm-up / calibration / streaming frame counts — here the threshold is fit on the validation split and metrics are reported on the held-out test split.

In [ ]:
PER_MODEL_COLS = [
    "threshold_value", "threshold_mode",
    "train_samples", "val_samples", "test_samples",
    "auroc", "aupr", "precision", "recall", "f1", "accuracy",
    "mean_score_ok", "mean_score_ng",
    "mean_latency_ms", "throughput_fps",
]

for model in MODELS_PRESENT:
    sub = (
        df[df["model"] == model]
        .set_index("category")[PER_MODEL_COLS]
        .sort_index()
    )
    print(f"\n=== {model} ({len(sub)} runs) ===")
    display(sub)

## 5. Heatmap — AUROC (category × model)

In [ ]:
def metric_heatmap(metric: str, title: str, *, vmin: float = 0.0, vmax: float = 1.0) -> None:
    mat = (
        df.pivot_table(index="category", columns="model", values=metric)
        .reindex(index=REAL_IAD_CATEGORIES, columns=ALL_MODELS)
    )
    fig, ax = plt.subplots(
        figsize=(1.1 * len(ALL_MODELS) + 2, 0.32 * len(REAL_IAD_CATEGORIES) + 1.5)
    )
    masked = np.ma.masked_invalid(mat.values)
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad(color="#e5e5e5")
    im = ax.imshow(masked, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(ALL_MODELS)))
    ax.set_xticklabels(ALL_MODELS, rotation=45, ha="right")
    ax.set_yticks(range(len(REAL_IAD_CATEGORIES)))
    ax.set_yticklabels(REAL_IAD_CATEGORIES)
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            if pd.notna(v):
                ax.text(
                    j, i, f"{v:.2f}",
                    ha="center", va="center", fontsize=7,
                    color="white" if v < (vmin + vmax) / 2 else "black",
                )
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    fig.tight_layout()
    plt.show()


metric_heatmap("auroc", "AUROC \u2014 category \u00d7 model")

## 6. Heatmap — F1 (category × model)

In [ ]:
metric_heatmap("f1", "F1 \u2014 category \u00d7 model")

## 7. Offline test-set scores — OK vs NG per (model, category)

Scatter of the per-image anomaly score over the held-out **test split** of each run (no warm-up / no calibration phase in offline evaluation). Green = OK ground truth, red = NG. The dashed line marks the threshold fit on the validation split (`val_f1` by default).

In [ ]:
SCATTER_COLS = 3

for model in MODELS_PRESENT:
    sub = df[df["model"] == model].sort_values("category").reset_index(drop=True)
    if sub.empty:
        continue
    n = len(sub)
    nrows = math.ceil(n / SCATTER_COLS)
    fig, axes = plt.subplots(
        nrows, SCATTER_COLS,
        figsize=(SCATTER_COLS * 4.2, nrows * 2.8),
        squeeze=False,
    )
    fig.suptitle(f"{model} ? offline test-set scores (OK vs NG) per category", fontsize=12)
    for i, row in sub.iterrows():
        ax = axes[i // SCATTER_COLS][i % SCATTER_COLS]
        recs = score_df[
            (score_df["experiment"] == row["experiment"])
            & (score_df["model"] == row["model"])
        ].sort_values("sample_idx")
        if recs.empty:
            ax.set_visible(False)
            continue
        scores = recs["score"].to_numpy(dtype=float)
        labels = recs["label"].to_numpy()
        idx = recs["sample_idx"].to_numpy(dtype=int)
        ok = labels == 0
        ng = labels == 1
        ax.scatter(idx[ok], scores[ok], s=8, c="tab:green", alpha=0.55, label="OK")
        ax.scatter(idx[ng], scores[ng], s=8, c="tab:red", alpha=0.55, label="NG")
        thr = row["threshold_value"]
        if pd.notna(thr):
            ax.axhline(thr, color="black", lw=0.7, ls="--", label=f"thr={thr:.2f}")
        ax.set_title(f"{row['category']} [{row['source_format']}]", fontsize=9)
        ax.set_xlabel("test sample index", fontsize=8)
        ax.set_ylabel("score", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6, loc="best")
        ax.grid(alpha=0.25)
    for j in range(n, nrows * SCATTER_COLS):
        axes[j // SCATTER_COLS][j % SCATTER_COLS].set_visible(False)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()
